<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-01-classical-ml-statsrefresher/ml-patterns/notebooks/exercises-0_5_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Lesson 0.5.1: How ML Finds Patterns — Practice Exercises

**Netsetos GenAI Course** | Module 0.5 — Classical ML & Stats Refresher

6 exercises covering supervised learning, unsupervised clustering, self-supervised next-token prediction, gradient descent, loss function comparison, and a full pipeline combining all 3 paradigms.

**The Universal ML Recipe:** Parameterize → Loss → Gradient Descent. Same recipe, 20 params to 1.8T params.

---

## Exercise 1 (Easy): Supervised Classifier — RandomForest

### 📚 Theory
Supervised = labeled data → train → predict. RandomForest = 100 decision trees voting. `feature_importances_` shows which features matter.

### 📋 Task
1. Generate 2000 samples with `make_classification(n_features=20, n_informative=12)`
2. Train RF, print accuracy, top-3 features, confusion matrix
3. Compare n_estimators: 10, 50, 100, 500

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: train RF, print accuracy, top-3 features
# TODO: compare n_estimators
# TODO: confusion matrix

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Train and evaluate
rf = RandomForestClassifier(100, random_state=42)
rf.fit(X_tr, y_tr)
acc = accuracy_score(y_te, rf.predict(X_te))
top3 = rf.feature_importances_.argsort()[-3:][::-1]
print(f'Accuracy: {acc:.4f}')
print(f'Top-3 features: {top3}')
print(f'Their importances: {rf.feature_importances_[top3].round(3)}')

# Compare n_estimators
print('\n--- n_estimators comparison ---')
for n in [10, 50, 100, 500]:
    m = RandomForestClassifier(n, random_state=42)
    m.fit(X_tr, y_tr)
    print(f'  n={n:3d}: accuracy={accuracy_score(y_te, m.predict(X_te)):.4f}')

# Confusion matrix
print(f'\nConfusion Matrix:\n{confusion_matrix(y_te, rf.predict(X_te))}')

</details>

---

## Exercise 2 (Easy): Unsupervised Clustering — K-Means

### 📚 Theory
K-Means: assign points to nearest centroid, update centroids, repeat. Elbow method: plot inertia vs K. Silhouette: -1 to 1 (higher = better separated). K-Means powers FAISS IVF indexing in every RAG pipeline.

### 📋 Task
1. Generate 3 clusters with `make_blobs(n_samples=500, centers=3)`
2. Run K-Means for K=2,3,4,5 — track inertia and silhouette
3. Identify optimal K, print cluster centers and sizes

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

X, y_true = make_blobs(n_samples=500, centers=3, random_state=42)

# TODO: K=2,3,4,5 — inertia + silhouette
# TODO: optimal K, cluster centers and sizes

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import numpy as np

X, y_true = make_blobs(n_samples=500, centers=3, random_state=42)

print('--- Elbow Method + Silhouette ---')
best_k, best_sil = 2, -1
for k in [2, 3, 4, 5]:
    km = KMeans(k, random_state=42, n_init=10)
    km.fit(X)
    sil = silhouette_score(X, km.labels_)
    marker = ' ← best' if sil > best_sil else ''
    if sil > best_sil:
        best_sil = sil
        best_k = k
    print(f'  K={k}: inertia={km.inertia_:8.0f}, silhouette={sil:.3f}{marker}')

print(f'\nOptimal K={best_k} (highest silhouette={best_sil:.3f})')

# Cluster details for optimal K
km_best = KMeans(best_k, random_state=42, n_init=10).fit(X)
unique, counts = np.unique(km_best.labels_, return_counts=True)
print(f'\nCluster sizes: {dict(zip(unique, counts))}')
print(f'Cluster centers (first 2 dims):')
for i, c in enumerate(km_best.cluster_centers_):
    print(f'  Cluster {i}: [{c[0]:.2f}, {c[1]:.2f}]')

</details>

---

## Exercise 3 (Medium): Self-Supervised — Next-Token Predictor

### 📚 Theory
GPT learns by predicting the next token from context. A bigram model is the simplest version: given current word → predict next from frequency table → softmax + temperature → sample. This IS how LLMs work, just at trillion-parameter scale.

### 📋 Task
1. Build a corpus of 10+ sentences about AI/ML
2. Build bigram frequency table
3. Convert to probabilities with temperature
4. Generate 20-token sequences at T=0.1, 0.5, 1.0, 2.0

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
import numpy as np
from collections import defaultdict

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat is happy today",
    "the dog is happy today",
    "a model learns from data",
    "a model predicts the output",
    "data drives the model forward",
    "the model is trained on data",
    "learning from data is powerful",
    "the output depends on the input",
]

# TODO: build bigram table
# TODO: sample with temperature
# TODO: generate sequences at 4 temperatures

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from collections import defaultdict

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat is happy today",
    "the dog is happy today",
    "a model learns from data",
    "a model predicts the output",
    "data drives the model forward",
    "the model is trained on data",
    "learning from data is powerful",
    "the output depends on the input",
]

# Build bigram table
bigrams = defaultdict(lambda: defaultdict(int))
all_words = set()
for sentence in corpus:
    words = sentence.split()
    all_words.update(words)
    for i in range(len(words) - 1):
        bigrams[words[i]][words[i + 1]] += 1

print(f'Vocabulary: {len(all_words)} words')
print(f'Bigram entries: {sum(len(v) for v in bigrams.values())}')

# Sampling with temperature
def sample_next(word, temperature=1.0):
    if word not in bigrams or len(bigrams[word]) == 0:
        return np.random.choice(list(all_words))
    next_words = list(bigrams[word].keys())
    counts = np.array([bigrams[word][w] for w in next_words], dtype=float)
    logits = np.log(counts + 1e-8)
    scaled = logits / max(temperature, 1e-8)
    probs = np.exp(scaled - np.max(scaled))
    probs = probs / probs.sum()
    return np.random.choice(next_words, p=probs)

def generate(start_word, length=20, temperature=1.0):
    tokens = [start_word]
    for _ in range(length - 1):
        tokens.append(sample_next(tokens[-1], temperature))
    return ' '.join(tokens)

# Generate at different temperatures
np.random.seed(42)
for temp in [0.1, 0.5, 1.0, 2.0]:
    text = generate('the', 20, temp)
    print(f'T={temp}: {text}')

</details>

---

## Exercise 4 (Medium): Gradient Descent from Scratch

### 📚 Theory
The universal engine: loss → gradient → update → repeat. Learning rate controls step size. Same algorithm for 2 parameters (linear regression) and 1.8 trillion (GPT-4).

### 📋 Task
1. Generate y = 3.5x + 2.0 + noise
2. Implement GD manually: w, b starting from 0
3. Compare lr = 0.001, 0.01, 0.1, 0.5
4. Identify which converges, which diverges

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
import numpy as np

np.random.seed(42)
X = np.random.randn(200)
y_true = 3.5 * X + 2.0 + np.random.randn(200) * 0.5

# TODO: GD for each learning rate
# TODO: compare convergence

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

np.random.seed(42)
X = np.random.randn(200)
y_true = 3.5 * X + 2.0 + np.random.randn(200) * 0.5

print('--- Gradient Descent: Learning Rate Comparison ---')
for lr in [0.001, 0.01, 0.1, 0.5]:
    w, b = 0.0, 0.0
    for epoch in range(100):
        y_pred = w * X + b
        loss = np.mean((y_pred - y_true) ** 2)
        dw = (2 / len(X)) * np.sum((y_pred - y_true) * X)
        db = (2 / len(X)) * np.sum(y_pred - y_true)
        w -= lr * dw
        b -= lr * db

    if np.isnan(loss):
        status = 'DIVERGED ❌'
    elif abs(w - 3.5) < 0.1 and abs(b - 2.0) < 0.1:
        status = 'CONVERGED ✅'
    else:
        status = 'TOO SLOW ⏳'

    print(f'  lr={lr:<5}: w={w:>8.4f}, b={b:>8.4f}, loss={loss:>10.4f}  {status}')

print(f'\n  True values: w=3.5, b=2.0')

</details>

---

## Exercise 5 (Hard): Loss Function Comparison

### 📚 Theory
Same data + different loss = different patterns. MSE: penalizes large errors quadratically (sensitive to outliers). MAE: penalizes linearly (robust). Huber: MSE for small errors, MAE for large (best of both).

### 📋 Task
1. Generate y = 3.5x + 2.0 + noise, add 5% outliers
2. Implement GD with MSE, MAE, and Huber loss
3. Compare: which recovers true parameters best with outliers?

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
import numpy as np

np.random.seed(42)
N = 200
X = np.random.randn(N)
y = 3.5 * X + 2.0 + np.random.randn(N) * 0.5

# Add 5% outliers
idx = np.random.choice(N, size=int(N*0.05), replace=False)
y[idx] += np.random.randn(len(idx)) * 20

# TODO: GD with MSE, MAE, Huber
# TODO: comparison table

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

np.random.seed(42)
N = 200
X = np.random.randn(N)
y = 3.5 * X + 2.0 + np.random.randn(N) * 0.5

# Add 5% outliers
idx = np.random.choice(N, size=int(N * 0.05), replace=False)
y[idx] += np.random.randn(len(idx)) * 20
print(f'Data: {N} points, {len(idx)} outliers\n')

def train_mse(X, y, lr=0.01, epochs=200):
    w, b = 0.0, 0.0
    for _ in range(epochs):
        pred = w * X + b
        error = pred - y
        w -= lr * (2 / N) * np.sum(error * X)
        b -= lr * (2 / N) * np.sum(error)
    loss = np.mean((w * X + b - y) ** 2)
    return w, b, loss

def train_mae(X, y, lr=0.001, epochs=500):
    w, b = 0.0, 0.0
    for _ in range(epochs):
        pred = w * X + b
        error = pred - y
        sign_e = np.sign(error)
        w -= lr * np.mean(sign_e * X)
        b -= lr * np.mean(sign_e)
    loss = np.mean(np.abs(w * X + b - y))
    return w, b, loss

def train_huber(X, y, delta=1.0, lr=0.01, epochs=300):
    w, b = 0.0, 0.0
    for _ in range(epochs):
        pred = w * X + b
        error = pred - y
        mask = np.abs(error) <= delta
        grad_w = np.where(mask, error * X, delta * np.sign(error) * X).mean()
        grad_b = np.where(mask, error, delta * np.sign(error)).mean()
        w -= lr * grad_w
        b -= lr * grad_b
    # Huber loss value
    error = w * X + b - y
    loss = np.where(np.abs(error) <= delta,
                    0.5 * error ** 2,
                    delta * (np.abs(error) - 0.5 * delta)).mean()
    return w, b, loss

results = {
    'MSE':   train_mse(X, y),
    'MAE':   train_mae(X, y),
    'Huber': train_huber(X, y),
}

print(f'{"Loss":<8} {"w (true=3.5)":>14} {"b (true=2.0)":>14} {"Final Loss":>12}')
print('-' * 52)
for name, (w, b, loss) in results.items():
    w_err = abs(w - 3.5)
    b_err = abs(b - 2.0)
    marker = ' ← best' if name == 'Huber' else (' ← outlier-pulled' if name == 'MSE' else '')
    print(f'  {name:<6} {w:>13.4f} {b:>13.4f} {loss:>11.4f}{marker}')

print(f'\nMSE pulled by outliers. MAE/Huber recover true params better.')

</details>

---

## Exercise 6 (Hard): Full ML Pipeline — All 3 Paradigms

### 📚 Theory
Real pipelines combine paradigms: unsupervised (cluster) → supervised (classify) → self-supervised-style (probability distribution). This mirrors GenAI: embedding space → retrieval → generation.

### 📋 Task
1. `make_blobs(centers=4)` → K-Means → silhouette
2. RF on K-Means labels → accuracy
3. `predict_proba` → probability distribution → temperature sampling
4. Cross-entropy loss
5. Unified report

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score

X, _ = make_blobs(n_samples=1000, centers=4, n_features=10, random_state=42)

# TODO: K-Means clustering
# TODO: RF classification on cluster labels
# TODO: probability distribution + temperature
# TODO: cross-entropy loss
# TODO: unified report

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score

X, _ = make_blobs(n_samples=1000, centers=4, n_features=10, random_state=42)

print('═' * 45)
print('  FULL ML PIPELINE — ALL 3 PARADIGMS')
print('═' * 45)

# Step 1: Unsupervised — K-Means
km = KMeans(4, random_state=42, n_init=10).fit(X)
labels = km.labels_
sil = silhouette_score(X, labels)
print(f'\n1. Unsupervised (K-Means):')
print(f'   Clusters: 4, Silhouette: {sil:.3f}')
unique, counts = np.unique(labels, return_counts=True)
print(f'   Sizes: {dict(zip(unique, counts))}')

# Step 2: Supervised — RF on K-Means labels
X_tr, X_te, y_tr, y_te = train_test_split(X, labels, test_size=0.2, random_state=42)
rf = RandomForestClassifier(100, random_state=42)
rf.fit(X_tr, y_tr)
acc = accuracy_score(y_te, rf.predict(X_te))
print(f'\n2. Supervised (RandomForest):')
print(f'   Accuracy: {acc:.4f}')

# Step 3: Self-supervised-style — probability distribution
probs = rf.predict_proba(X_te)
print(f'\n3. Probability Distribution:')
for temp in [0.1, 1.0, 2.0]:
    # Apply temperature to log-probabilities
    log_p = np.log(probs[0] + 1e-10)
    scaled = np.exp(log_p / temp)
    scaled = scaled / scaled.sum()
    sampled = np.random.choice(4, p=scaled)
    print(f'   T={temp}: probs={scaled.round(3)}, sampled=cluster_{sampled}')

# Step 4: Cross-entropy loss
correct_probs = probs[np.arange(len(y_te)), y_te]
ce_loss = -np.mean(np.log(correct_probs + 1e-10))
print(f'\n4. Cross-Entropy Loss: {ce_loss:.4f}')

print(f'\n{"═" * 45}')
print(f'  All 3 paradigms combined successfully!')
print(f'  Silhouette={sil:.3f}, Accuracy={acc:.4f}, CE={ce_loss:.4f}')
print(f'{"═" * 45}')

</details>

---

## 🎉 Done!

You've built all 3 ML paradigms from scratch:
- **Supervised** — labeled classification with RandomForest
- **Unsupervised** — K-Means clustering with silhouette scoring
- **Self-supervised** — next-token prediction with bigram model + temperature
- **Gradient Descent** — the universal engine, from scratch
- **Loss Functions** — MSE vs MAE vs Huber on outlier data
- **Full Pipeline** — all paradigms combined

The universal recipe — **parameterize, loss, gradient descent** — works for 2 parameters or 1.8 trillion. Next: **Lesson 0.5.2.**